# 🎧 Dataset Explorer — Speech-to-Speech Audio Pairs

Browse and listen to aligned **English ↔ German** (or any other supported language pair) audio samples from:

| Dataset | Type | Lang pairs | Notes |
|---|---|---|---|
| **FLEURS** | Curated, human-labelled | en↔de (+ 100 others) | High quality; gold standard |
| **SeamlessAlign** | Web-mined, SONAR-filtered | 35 pairs × en | Large scale; needs paired LID |
| **SeamlessAlign-Expressive** | Web-mined, expressive subset | de/es/fr/it/zh × en | Smaller, more expressive speech |

---
**How it works:** `load_data()` loads both sides of each pair, runs quality filters (duration + silence) and paired Language ID (Whisper-tiny) on SeamlessAlign data, then aligns by common IDs so that `datasets['en'][i]` and `datasets['de'][i]` are always a translation pair.

## 0. Environment Setup

In [ ]:
import os, sys
from pathlib import Path

# ── Resolve project root from notebook location ──────────────────────────────
NOTEBOOK_DIR = Path(os.getcwd()).resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent   # Speech-To-Speech-Model/

for p in [str(PROJECT_ROOT)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"Project root : {PROJECT_ROOT}")
print(f"Python path  : {[p for p in sys.path if 'Speech' in p]}")

In [ ]:
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display, Audio, HTML
import ipywidgets as widgets

# Project imports
from dataset_loader import (
    load_data,
    play_audio,
    save_audio,
    SEAMLESS_PAIR_MAPPING,
    SEAMLESS_EXPRESSIVE_PAIR_MAPPING,
    _SEAMLESS_S2S_LANGS,
    _SEAMLESS_EXPRESSIVE_LANGS,
    MIN_DURATION_S,
    MAX_DURATION_S,
)

print("✅ Imports OK")
print(f"SeamlessAlign full   : {len(SEAMLESS_PAIR_MAPPING)} language pairs")
print(f"SeamlessAlign express: {len(SEAMLESS_EXPRESSIVE_PAIR_MAPPING)} language pairs")
print(f"Supported (full)     : {sorted(_SEAMLESS_S2S_LANGS)}")

---
## 1. FLEURS — en ↔ de (Gold Standard)

FLEURS is a human-curated, human-labelled benchmark. No LID filtering needed.
Loads quickly (~2k paired samples for `train` split).

In [ ]:
# ── Load FLEURS en ↔ de ──────────────────────────────────────────────────────
fleurs = load_data(
    dataset=['fleurs'],
    lang=['en', 'de'],
    split='train',
    num_samples=50,        # ← increase to load more; None = all ~2k pairs
)

en_fleurs = fleurs.get('en')
de_fleurs = fleurs.get('de')

print(f"\n✅ FLEURS loaded")
print(f"   EN: {len(en_fleurs)} samples")
print(f"   DE: {len(de_fleurs)} samples")
print(f"   Columns: {en_fleurs.column_names}")
print(f"\nSample EN[0]:")
print(f"   id          : {en_fleurs[0]['id']}")
print(f"   transcription: {en_fleurs[0]['transcription'][:80]}...")
print(f"   duration    : {len(en_fleurs[0]['audio']['array']) / en_fleurs[0]['audio']['sampling_rate']:.2f}s")
print(f"   gender      : {en_fleurs[0]['gender']}")

In [ ]:
# ── Interactive FLEURS Sample Browser ────────────────────────────────────────
def browse_pair(datasets_dict, src_lang='en', tgt_lang='de', title='Dataset'):
    """
    Renders an interactive slider to browse aligned audio pairs.
    Press the slider or use arrow keys to step through samples.
    """
    src_ds = datasets_dict.get(src_lang)
    tgt_ds = datasets_dict.get(tgt_lang)
    if src_ds is None or tgt_ds is None or len(src_ds) == 0:
        print(f"No data for {src_lang}/{tgt_lang}")
        return

    n = min(len(src_ds), len(tgt_ds))

    slider = widgets.IntSlider(
        value=0, min=0, max=n - 1, step=1,
        description='Sample #',
        layout=widgets.Layout(width='80%'),
        style={'description_width': '80px'},
    )
    out = widgets.Output()

    def _render(idx):
        src_rec = src_ds[idx]
        tgt_rec = tgt_ds[idx]

        src_arr = np.array(src_rec['audio']['array'], dtype=np.float32)
        tgt_arr = np.array(tgt_rec['audio']['array'], dtype=np.float32)
        src_sr  = src_rec['audio']['sampling_rate']
        tgt_sr  = tgt_rec['audio']['sampling_rate']
        src_dur = len(src_arr) / src_sr
        tgt_dur = len(tgt_arr) / tgt_sr

        out.clear_output(wait=True)
        with out:
            display(HTML(f"""
            <div style='font-family:monospace; background:#1e1e2e; color:#cdd6f4;
                        padding:14px; border-radius:10px; margin:6px 0'>
              <b style='color:#89b4fa; font-size:1.1em'>🎙️ {title} — Sample {idx+1}/{n}</b><br><br>
              <span style='color:#a6e3a1'>ID:</span> {src_rec['id']}<br>
              <span style='color:#a6e3a1'>Gender:</span> {src_rec.get('gender','unknown')}<br><br>
              <span style='color:#fab387'>🇬🇧 {src_lang.upper()} ({src_dur:.2f}s):</span><br>
              <span style='color:#cdd6f4; font-style:italic'>{src_rec.get('transcription','(no transcription)') or '(no transcription)'}</span><br><br>
              <span style='color:#f38ba8'>🇩🇪 {tgt_lang.upper()} ({tgt_dur:.2f}s):</span><br>
              <span style='color:#cdd6f4; font-style:italic'>{tgt_rec.get('transcription','(no transcription)') or '(no transcription)'}</span>
            </div>
            """))

            # Waveform plots side by side
            fig, axes = plt.subplots(1, 2, figsize=(14, 2.5),
                                     facecolor='#1e1e2e')
            for ax, arr, sr, lang, colour, flag in [
                (axes[0], src_arr, src_sr, src_lang, '#89b4fa', '🇬🇧'),
                (axes[1], tgt_arr, tgt_sr, tgt_lang, '#f38ba8', '🇩🇪'),
            ]:
                t = np.linspace(0, len(arr) / sr, num=len(arr))
                ax.plot(t, arr, color=colour, linewidth=0.6, alpha=0.9)
                ax.set_facecolor('#181825')
                ax.tick_params(colors='#6c7086', labelsize=8)
                ax.set_xlabel('Time (s)', color='#6c7086', fontsize=8)
                ax.set_title(f'{flag} {lang.upper()} waveform', color=colour, fontsize=9)
                for spine in ax.spines.values():
                    spine.set_edgecolor('#313244')
            plt.tight_layout(pad=1.0)
            plt.show()

            print(f"▶ {src_lang.upper()} audio:")
            display(Audio(data=src_arr, rate=src_sr))
            print(f"▶ {tgt_lang.upper()} audio:")
            display(Audio(data=tgt_arr, rate=tgt_sr))

    widgets.interactive_output(_render, {'idx': slider})
    _render(0)   # show first sample immediately

    display(widgets.VBox(
        [slider, out],
        layout=widgets.Layout(width='100%')
    ))


browse_pair(fleurs, src_lang='en', tgt_lang='de', title='FLEURS')

In [ ]:
# ── FLEURS dataset statistics ─────────────────────────────────────────────────
def show_stats(datasets_dict, title='Dataset'):
    print(f"\n{'='*55}")
    print(f" {title} Statistics")
    print(f"{'='*55}")
    for lang, ds in datasets_dict.items():
        if ds is None or len(ds) == 0:
            print(f"  {lang}: empty")
            continue
        durations = [
            len(r['audio']['array']) / r['audio']['sampling_rate']
            for r in ds
        ]
        total_h = sum(durations) / 3600
        print(f"  {lang.upper()}: {len(ds):>5} samples | "
              f"total {total_h:.2f}h | "
              f"avg {np.mean(durations):.1f}s | "
              f"range [{min(durations):.1f}s, {max(durations):.1f}s]")

    # Duration histogram
    fig, axes = plt.subplots(1, len(datasets_dict), figsize=(6 * len(datasets_dict), 3),
                             facecolor='#1e1e2e')
    if len(datasets_dict) == 1:
        axes = [axes]
    colours = ['#89b4fa', '#f38ba8', '#a6e3a1', '#fab387', '#cba6f7']
    for ax, (lang, ds), colour in zip(axes, datasets_dict.items(), colours):
        if ds is None or len(ds) == 0:
            continue
        durs = [len(r['audio']['array']) / r['audio']['sampling_rate'] for r in ds]
        ax.hist(durs, bins=30, color=colour, edgecolor='#313244', alpha=0.85)
        ax.axvline(MIN_DURATION_S, color='#f9e2af', linestyle='--', linewidth=1,
                   label=f'min filter ({MIN_DURATION_S}s)')
        ax.axvline(MAX_DURATION_S, color='#f9e2af', linestyle=':',  linewidth=1,
                   label=f'max filter ({MAX_DURATION_S}s)')
        ax.set_facecolor('#181825')
        ax.tick_params(colors='#6c7086', labelsize=8)
        ax.set_xlabel('Duration (s)', color='#6c7086', fontsize=8)
        ax.set_ylabel('Count',        color='#6c7086', fontsize=8)
        ax.set_title(f'{lang.upper()} duration distribution', color=colour, fontsize=9)
        ax.legend(fontsize=7, facecolor='#313244', labelcolor='#cdd6f4')
        for spine in ax.spines.values():
            spine.set_edgecolor('#313244')
    fig.suptitle(title, color='#cdd6f4', fontsize=11, y=1.02)
    plt.tight_layout()
    plt.show()

show_stats(fleurs, title='FLEURS en↔de')

---
## 2. SeamlessAlign — Full 35-pair (en ↔ de) with Paired LID

Web-scraped data — language labels are **not** ground truth. Three layers of filtering are applied automatically:
1. **Quality pre-filter** — duration gates (1–20s) + silence RMS check (cheap, CPU)
2. **Paired LID** — Whisper-tiny on both sides simultaneously. Only pairs where **both** src and tgt match their expected language are kept.
3. **Common-ID intersection** — final alignment ensures `en[i]` and `de[i]` are always the same pair.

> ⚠️ Paired LID is slow (Whisper on every sample). Set `seamless_paired_lid=False` for a quick debug run.

In [ ]:
# ── Load SeamlessAlign en ↔ de with paired LID ───────────────────────────────
# For a quick sanity-check, set num_samples=20 and seamless_paired_lid=False.
# For a real training run, use num_samples=None and seamless_paired_lid=True.

seamless = load_data(
    dataset=['seamless_align'],
    lang=['en', 'de'],
    split='train',
    num_samples=100,              # ← set to None for full dataset
    seamless_tgt_lid=True,        # ← catches en-> en (same OR diff speakers)
    seamless_paired_lid=False,     # ← highly recommended for web-scraped data
    seamless_expressive=True,     # ← False = full 35-pair dataset
)

en_seamless = seamless.get('en')
de_seamless = seamless.get('de')

print(f"\n✅ SeamlessAlign loaded")
print(f"   EN: {len(en_seamless)} samples after LID filtering")
print(f"   DE: {len(de_seamless)} samples after LID filtering")

In [ ]:
# ── Browse SeamlessAlign pairs ────────────────────────────────────────────────
browse_pair(seamless, src_lang='en', tgt_lang='de', title='SeamlessAlign (full)')

In [ ]:
show_stats(seamless, title='SeamlessAlign en↔de')

---
## 3. SeamlessAlign-Expressive — 5-pair Subset

The expressive subset covers only: **de / es / fr / it / zh** paired with **en**.
Smaller but reportedly more expressive and prosodically richer than the full set.

In [ ]:
# ── SeamlessAlign-Expressive: fr ↔ en ────────────────────────────────────────
# Change lang=['fr', 'en'] to any of: de, es, it, zh + en

expressive = load_data(
    dataset=['seamless_align'],
    lang=['de', 'en'],
    split='train',
    num_samples=20,
    seamless_expressive=True,     # ← use the 5-pair expressive variant
    seamless_paired_lid=True,
)

print(f"Expressive EN: {len(expressive.get('en', []))} | FR: {len(expressive.get('fr', []))}")

In [ ]:
browse_pair(expressive, src_lang='en', tgt_lang='fr', title='SeamlessAlign-Expressive')

---
## 4. Mixed Dataset — FLEURS + SeamlessAlign

Combine both sources. `load_data` concatenates them and runs deduplication by ID before returning.

In [ ]:
# ── Mixed: FLEURS + SeamlessAlign en ↔ de ───────────────────────────────────
mixed = load_data(
    dataset=['fleurs', 'seamless_align'],
    lang=['en', 'de'],
    split='train',
    num_samples=60,
    seamless_paired_lid=True,
)

print(f"\n✅ Mixed dataset loaded")
print(f"   EN: {len(mixed.get('en', []))} total (FLEURS + SeamlessAlign)")
print(f"   DE: {len(mixed.get('de', []))} total")

In [ ]:
browse_pair(mixed, src_lang='en', tgt_lang='de', title='Mixed (FLEURS + SeamlessAlign)')

In [ ]:
show_stats(mixed, title='Mixed en↔de')

---
## 5. Quick-Load Utility (No LID — for debugging)

Use this cell when you want to **quickly inspect raw data** without waiting for LID.
⚠️ Do NOT use for training — unfiltered SeamlessAlign contains noisy/mislabelled samples.

In [ ]:
# ── Fast debug load: no LID, small batch ────────────────────────────────────
debug = load_data(
    dataset=['seamless_align'],
    lang=['en', 'de'],
    split='train',
    num_samples=10,
    seamless_paired_lid=False,    # ← skip LID for speed
)

print(f"Debug: EN={len(debug.get('en',[]))}, DE={len(debug.get('de',[]))}")
browse_pair(debug, src_lang='en', tgt_lang='de', title='Debug (no LID)')

---
## 6. Supported Language Pairs Reference

In [ ]:
# ── Print all supported SeamlessAlign configs ────────────────────────────────
from dataset_loader import SEAMLESS_PAIR_MAPPING, SEAMLESS_EXPRESSIVE_PAIR_MAPPING

print("SeamlessAlign FULL (35 pairs × en)")
print("─" * 55)
for i, (key, cfg) in enumerate(sorted(SEAMLESS_PAIR_MAPPING.items()), 1):
    print(f"  [{i:02d}] {key[0]} ↔ {key[1]}  →  config: {cfg}")

print()
print("SeamlessAlign EXPRESSIVE (5 pairs × en)")
print("─" * 55)
for key, cfg in sorted(SEAMLESS_EXPRESSIVE_PAIR_MAPPING.items()):
    print(f"  {key[0]} ↔ {key[1]}  →  config: {cfg}")

print()
print("Usage examples:")
print("""
# Full 35-pair with paired LID (recommended for training)
datasets = load_data(
    lang=['de', 'en'], dataset=['seamless_align'],
    seamless_paired_lid=True,
)

# Mix with FLEURS for cleaner en side
datasets = load_data(
    lang=['de', 'en'], dataset=['fleurs', 'seamless_align'],
)

# Expressive 5-pair variant
datasets = load_data(
    lang=['fr', 'en'], dataset=['seamless_align'],
    seamless_expressive=True,
)

# Any of the 35 supported pairs
datasets = load_data(
    lang=['ja', 'en'], dataset=['seamless_align'],
)
""")

---
## 7. Single-Sample Playback Helpers

In [ ]:
# ── Play any sample by index from any dataset ────────────────────────────────
def play_pair(datasets_dict, idx=0, src_lang='en', tgt_lang='de'):
    """Print metadata and play audio for a single pair by index."""
    src = datasets_dict.get(src_lang)
    tgt = datasets_dict.get(tgt_lang)
    if src is None or tgt is None:
        print(f"No data for {src_lang}/{tgt_lang}")
        return
    if idx >= len(src):
        print(f"Index {idx} out of range (max {len(src)-1})")
        return

    s, t = src[idx], tgt[idx]
    dur_s = len(s['audio']['array']) / s['audio']['sampling_rate']
    dur_t = len(t['audio']['array']) / t['audio']['sampling_rate']

    display(HTML(f"""
    <div style='font-family:monospace; background:#1e1e2e; color:#cdd6f4;
                padding:12px; border-radius:8px'>
      <b style='color:#89b4fa'>Sample index {idx}</b>  |  ID: {s['id']}
      | gender: {s.get('gender','unknown')}<br><br>
      <span style='color:#fab387'>🇬🇧 {src_lang.upper()} ({dur_s:.2f}s):</span>
      {s.get('transcription','') or '<i>(no transcription)</i>'}<br><br>
      <span style='color:#f38ba8'>🇩🇪 {tgt_lang.upper()} ({dur_t:.2f}s):</span>
      {t.get('transcription','') or '<i>(no transcription)</i>'}
    </div>
    """))
    display(Audio(data=s['audio']['array'], rate=s['audio']['sampling_rate']))
    display(Audio(data=t['audio']['array'], rate=t['audio']['sampling_rate']))


# ── Example usage ─────────────────────────────────────────────────────────────
# Change fleurs → seamless or mixed, and idx to any sample number
play_pair(fleurs, idx=0, src_lang='en', tgt_lang='de')

In [ ]:
# ── Save a pair to disk ───────────────────────────────────────────────────────
idx = 0
save_audio(fleurs['en'][idx], path=str(PROJECT_ROOT / 'Notebooks' / 'sample_en.wav'))
save_audio(fleurs['de'][idx], path=str(PROJECT_ROOT / 'Notebooks' / 'sample_de.wav'))
print(f"Saved sample {idx} → sample_en.wav / sample_de.wav")